# Fixed-Income ETF Fragility Analysis

Three model sets:
- **M1** — weekly return sensitivity to macro shocks (H1, H2)
- **M2** — fragility signals predict forward downside outcomes (H3)
- **M3** — high-stress regime interactions (H4)

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import statsmodels.formula.api as smf

from src import config
from src.features.category import assign_category_bucket
from src.features.rolling_risk import add_rolling_risk_metrics
from src.features.forward_outcomes import add_forward_outcomes
from src.features.stress_index import add_stress_index

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 9,
})

Matplotlib is building the font cache; this may take a moment.


## 1. Load panel

In [2]:
panel = pd.read_csv(config.CORE_PANEL_CSV, parse_dates=['Date', 'Inception'])
panel = panel.sort_values(['Symbol', 'Date']).reset_index(drop=True)

# Apply feature engineering if the panel pre-dates the Phase 2 build.
# When core_panel.csv is regenerated via the full pipeline these become no-ops.
if 'category_bucket' not in panel.columns:
    panel = assign_category_bucket(panel)
if 'stress_index' not in panel.columns:
    panel = add_stress_index(panel)
if 'vol_12w' not in panel.columns:
    panel = add_rolling_risk_metrics(panel)
if 'fwd_ret_4w' not in panel.columns:
    panel = add_forward_outcomes(panel)

print(f'Rows: {len(panel):,}   ETFs: {panel.Symbol.nunique():,}   '
      f'Weeks: {panel.Date.nunique():,}')
print(f'Date range: {panel.Date.min().date()} to {panel.Date.max().date()}')
print(f'Columns: {list(panel.columns)}')

Rows: 156,409   ETFs: 347   Weeks: 520
Date range: 2016-04-22 to 2026-04-03
Columns: ['Date', 'Symbol', 'Return', 'ANFCI', 'BAMLC0A0CM', 'DGS10', 'T10Y2Y', 'T5YIE', 'VIX', 'MOVE', 'GPR', 'd_ANFCI', 'd_BAMLC0A0CM', 'd_DGS10', 'd_T10Y2Y', 'd_T5YIE', 'd_VIX', 'd_MOVE', 'd_GPR', 'RF_w', 'Name', 'Assets', 'ETF Database Category', 'ER', 'Inception', 'RET_XS', 'age_years', 'Assets_clean', 'ER_clean', 'log_assets', 'category_bucket', 'stress_index', 'high_stress', 'vol_12w', 'var_12w_90', 'downside_vol_12w', 'maxdd_12w', 'es_12w_90', 'fwd_ret_4w', 'fwd_maxdd_12w', 'fwd_vol_12w']


## 2. Universe overview

In [ ]:
# ETF count per category (unique symbols)
etf_counts = (
    panel.drop_duplicates('Symbol')
    .groupby('category_bucket', dropna=False)
    .size()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(6, 4))
etf_counts.plot.barh(ax=ax, color='steelblue', edgecolor='none')
ax.set_xlabel('Number of ETFs')
ax.set_title('ETF count by category bucket')
plt.tight_layout()
plt.show()

print(etf_counts.sort_values(ascending=False).to_string())

## 3. Summary statistics

In [ ]:
STAT_COLS = ['Return', 'RET_XS', 'vol_12w', 'downside_vol_12w', 'maxdd_12w',
             'var_12w_90', 'es_12w_90', 'log_assets', 'ER_clean', 'age_years']

def fmt_stats(df):
    return df[STAT_COLS].describe(percentiles=[.1, .25, .5, .75, .9]).T[
        ['count', 'mean', 'std', '10%', '25%', '50%', '75%', '90%']
    ].round(4)

print('--- Full panel ---')
display(fmt_stats(panel))

In [ ]:
# Category-level return and risk summary
CAT_METRICS = ['Return', 'RET_XS', 'vol_12w', 'maxdd_12w', 'es_12w_90']
cat_summary = (
    panel.groupby('category_bucket')[CAT_METRICS]
    .mean()
    .sort_values('vol_12w', ascending=False)
    .round(4)
)
display(cat_summary)

## 4. Macro shocks — correlation matrix

In [ ]:
MACRO_CHANGES = ['d_ANFCI', 'd_BAMLC0A0CM', 'd_DGS10', 'd_T10Y2Y',
                 'd_T5YIE', 'd_VIX', 'd_MOVE', 'd_GPR']

macro_weekly = panel[['Date'] + MACRO_CHANGES].drop_duplicates('Date')
corr = macro_weekly[MACRO_CHANGES].corr().round(2)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_xticks(range(len(MACRO_CHANGES)))
ax.set_yticks(range(len(MACRO_CHANGES)))
labels = [c.replace('d_', '') for c in MACRO_CHANGES]
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels)
for i in range(len(MACRO_CHANGES)):
    for j in range(len(MACRO_CHANGES)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center',
                fontsize=7, color='black')
ax.set_title('Macro shock correlation matrix (weekly changes)')
plt.tight_layout()
plt.show()

display(corr)

## 5. Stress index through time

In [ ]:
stress_ts = (
    panel[['Date', 'stress_index', 'high_stress']]
    .drop_duplicates('Date')
    .set_index('Date')
    .sort_index()
)

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(stress_ts.index, stress_ts['stress_index'], lw=0.9, color='steelblue')
ax.axhline(1.0, color='firebrick', lw=0.8, ls='--', label='high-stress threshold')
ax.fill_between(stress_ts.index, stress_ts['stress_index'],
                where=stress_ts['high_stress'] == 1,
                alpha=0.25, color='firebrick', label='high-stress weeks')
ax.set_ylabel('Stress index (z-score)')
ax.set_title('Composite macro stress index')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

n_stress = stress_ts['high_stress'].sum()
pct_stress = n_stress / len(stress_ts) * 100
print(f'High-stress weeks: {n_stress} of {len(stress_ts)} ({pct_stress:.1f}%)')

## 6. Structural variable distributions

In [ ]:
latest = panel.sort_values('Date').groupby('Symbol').tail(1)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, col, label in zip(
    axes,
    ['log_assets', 'ER_clean', 'age_years'],
    ['log(AUM)', 'Expense ratio', 'Age (years)']
):
    latest[col].dropna().hist(bins=25, ax=ax, color='steelblue', edgecolor='none')
    ax.set_title(label)
    ax.set_ylabel('ETFs')

plt.suptitle('Distribution of structural variables (latest observation per ETF)',
             y=1.02)
plt.tight_layout()
plt.show()

## 7. Model 1 — Macro shock sensitivity

Pooled OLS: `RET_XS ~ macro_changes + C(category_bucket)`.  
Tests H1 (heterogeneous macro sensitivity) and H2 (structure beyond category).  
Standard errors clustered by Symbol.

In [ ]:
# Drop Other / leveraged / inverse for this regression
reg_panel = panel.dropna(subset=['RET_XS', 'category_bucket'] + MACRO_CHANGES).copy()
reg_panel = reg_panel[reg_panel['category_bucket'] != 'Other']

formula_m1 = (
    'RET_XS ~ '
    'd_ANFCI + d_BAMLC0A0CM + d_DGS10 + d_T10Y2Y + d_T5YIE + '
    'd_VIX + d_MOVE + d_GPR + C(category_bucket)'
)

m1 = smf.ols(formula_m1, data=reg_panel).fit(
    cov_type='cluster', cov_kwds={'groups': reg_panel['Symbol']}
)
print(m1.summary())

In [ ]:
# Average macro betas by category from per-ETF time-series regressions
MACRO_SHOCKS = ['d_ANFCI', 'd_BAMLC0A0CM', 'd_DGS10', 'd_VIX', 'd_MOVE']

betas = []
for sym, grp in reg_panel.groupby('Symbol'):
    if len(grp) < 30:
        continue
    try:
        m = smf.ols(
            'RET_XS ~ ' + ' + '.join(MACRO_SHOCKS), data=grp
        ).fit()
        row = {'Symbol': sym,
               'category_bucket': grp['category_bucket'].iloc[0]}
        for s in MACRO_SHOCKS:
            row[s] = m.params.get(s, np.nan)
        betas.append(row)
    except Exception:
        pass

betas_df = pd.DataFrame(betas)

cat_betas = (
    betas_df.groupby('category_bucket')[MACRO_SHOCKS]
    .mean()
    .round(4)
)
print('Average macro betas by category:')
display(cat_betas)

In [ ]:
# Heatmap of category-average betas
fig, ax = plt.subplots(figsize=(7, 4))
vabs = cat_betas.abs().max().max()
im = ax.imshow(cat_betas.values, cmap='RdBu_r', vmin=-vabs, vmax=vabs,
               aspect='auto')
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(MACRO_SHOCKS)))
ax.set_xticklabels([c.replace('d_', '') for c in MACRO_SHOCKS], rotation=30, ha='right')
ax.set_yticks(range(len(cat_betas)))
ax.set_yticklabels(cat_betas.index)
for i in range(len(cat_betas)):
    for j in range(len(MACRO_SHOCKS)):
        ax.text(j, i, f'{cat_betas.iloc[i, j]:.3f}', ha='center', va='center',
                fontsize=7)
ax.set_title('Average macro beta by category')
plt.tight_layout()
plt.show()

## 8. Model 2 — Fragility predicts forward downside (H3)

OLS: `fwd_maxdd_12w ~ vol_12w + downside_vol_12w + maxdd_12w + log_assets + ER_clean + age_years + C(category_bucket)`.  
A negative coefficient on the fragility metrics means higher current fragility → worse (more negative) future drawdown.

In [ ]:
FRAG_VARS = ['vol_12w', 'downside_vol_12w', 'maxdd_12w']
STRUCT_VARS = ['log_assets', 'ER_clean', 'age_years']

reg2 = panel.dropna(
    subset=['fwd_maxdd_12w', 'category_bucket'] + FRAG_VARS + STRUCT_VARS
).copy()
reg2 = reg2[reg2['category_bucket'] != 'Other']

formula_m2 = (
    'fwd_maxdd_12w ~ '
    'vol_12w + downside_vol_12w + maxdd_12w + '
    'log_assets + ER_clean + age_years + '
    'C(category_bucket)'
)

m2 = smf.ols(formula_m2, data=reg2).fit(
    cov_type='cluster', cov_kwds={'groups': reg2['Symbol']}
)
print(m2.summary())

In [ ]:
# Also run with fwd_vol_12w as the outcome
reg2v = panel.dropna(
    subset=['fwd_vol_12w', 'category_bucket'] + FRAG_VARS + STRUCT_VARS
).copy()
reg2v = reg2v[reg2v['category_bucket'] != 'Other']

formula_m2v = (
    'fwd_vol_12w ~ '
    'vol_12w + downside_vol_12w + maxdd_12w + '
    'log_assets + ER_clean + age_years + '
    'C(category_bucket)'
)

m2v = smf.ols(formula_m2v, data=reg2v).fit(
    cov_type='cluster', cov_kwds={'groups': reg2v['Symbol']}
)
print(m2v.summary())

## 9. Model 3 — Regime interactions (H4)

Does `vol_12w` predicting worse forward outcomes amplify during high-stress regimes?  
Interaction term: `vol_12w * high_stress`.

In [ ]:
reg3 = panel.dropna(
    subset=['fwd_maxdd_12w', 'category_bucket', 'high_stress'] + FRAG_VARS + STRUCT_VARS
).copy()
reg3 = reg3[reg3['category_bucket'] != 'Other']

formula_m3 = (
    'fwd_maxdd_12w ~ '
    'vol_12w * high_stress + '
    'downside_vol_12w + maxdd_12w + '
    'log_assets + ER_clean + age_years + '
    'C(category_bucket)'
)

m3 = smf.ols(formula_m3, data=reg3).fit(
    cov_type='cluster', cov_kwds={'groups': reg3['Symbol']}
)
print(m3.summary())

## 10. Fragility decile sort — average forward drawdown by decile

In [ ]:
decile_data = panel.dropna(subset=['vol_12w', 'fwd_maxdd_12w']).copy()
decile_data = decile_data[decile_data['category_bucket'] != 'Other']

# Decile rank vol_12w within each week (cross-sectional sort)
decile_data['frag_decile'] = (
    decile_data.groupby('Date')['vol_12w']
    .transform(lambda x: pd.qcut(x, 10, labels=False, duplicates='drop') + 1)
)

decile_summary = (
    decile_data.groupby('frag_decile')[['fwd_maxdd_12w', 'fwd_vol_12w', 'fwd_ret_4w']]
    .mean()
    .round(4)
)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
for ax, col, title, color in zip(
    axes,
    ['fwd_maxdd_12w', 'fwd_vol_12w', 'fwd_ret_4w'],
    ['Avg fwd 12w max drawdown', 'Avg fwd 12w vol', 'Avg fwd 4w return'],
    ['firebrick', 'darkorange', 'steelblue']
):
    ax.bar(decile_summary.index, decile_summary[col], color=color, edgecolor='none')
    ax.set_xlabel('Fragility decile (vol_12w, 1=low)')
    ax.set_title(title)
    ax.axhline(0, color='black', lw=0.5)

plt.suptitle('Forward outcomes by fragility decile (cross-sectional sort on vol_12w)',
             y=1.02)
plt.tight_layout()
plt.show()

display(decile_summary)